In [ ]:
# Import necessary libraries
import argparse
import json
import logging
import sys
import time
import traceback
from pathlib import Path
from typing import Dict, List, Any, Optional
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add src to path for imports
sys.path.append(str(Path.cwd().parent))

# Import custom modules
from src.utils.config import setup_logging, load_config, get_data_path, get_model_path, get_results_path
from src.data.make_dataset import load_raw_data, load_processed_data
from src.data.preprocessing import preprocess_features, split_data
from src.models.train_model import train_models, ModelTrainer
from src.evaluation.metrics import calculate_comprehensive_metrics, compare_models
from src.visualization.model_plots import generate_evaluation_plots

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully!")
print(f"📅 Pipeline started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# Configuration setup
class MLPipeline:
    """Complete ML Pipeline for Healthcare Length of Stay Prediction"""
    
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.results = {}
        self.start_time = datetime.now()
        self.pipeline_id = f"pipeline_{self.start_time.strftime('%Y%m%d_%H%M%S')}"
        
        # Initialize paths
        self.data_path = get_data_path(config)
        self.model_path = get_model_path(config)
        self.results_path = get_results_path(config)
        
        logger.info(f"Pipeline initialized: {self.pipeline_id}")

# Load configuration (you may need to adjust the path)
try:
    config = load_config('config/config.yaml')
    print("✅ Configuration loaded successfully!")
except FileNotFoundError:
    # Create a basic config if file doesn't exist
    config = {
        'data': {
            'train_file': 'train.csv',
            'test_file': 'test.csv',
            'processed_data_path': 'data/processed'
        },
        'target_column': 'lengthofstay',
        'models': ['xgboost', 'lightgbm'],
        'cross_validation': {'n_splits': 5, 'random_state': 42}
    }
    print("⚠️  Using default configuration")

# Initialize pipeline
pipeline = MLPipeline(config)
print(f"🚀 Pipeline ID: {pipeline.pipeline_id}")
print(f"📁 Data path: {pipeline.data_path}")
print(f"🤖 Model path: {pipeline.model_path}")
print(f"📊 Results path: {pipeline.results_path}")


In [ ]:
def run_data_loading(pipeline, force_reload: bool = False):
    """Load and validate data."""
    print("=" * 50)
    print("STEP 2: DATA LOADING")
    print("=" * 50)
    
    try:
        step_start = time.time()
        
        # Check for processed data first
        processed_data_path = pipeline.data_path / 'processed'
        
        if not force_reload and (processed_data_path / 'X_preprocessed.csv').exists():
            print("📂 Loading existing processed data...")
            X, y = load_processed_data(pipeline.config)
            data_source = "processed"
        else:
            print("📂 Loading raw data...")
            train_data, test_data = load_raw_data(pipeline.config)
            
            # Combine for preprocessing
            if test_data is not None:
                combined_data = pd.concat([train_data, test_data], ignore_index=True)
                print(f"📊 Combined training and test data: {combined_data.shape}")
            else:
                combined_data = train_data
                print(f"📊 Using training data only: {combined_data.shape}")
            
            X = combined_data.drop(columns=[pipeline.config.get('target_column', 'lengthofstay')])
            y = combined_data[pipeline.config.get('target_column', 'lengthofstay')]
            data_source = "raw"
        
        step_time = time.time() - step_start
        
        # Data validation and summary
        print(f"\n📈 Data Summary:")
        print(f"   • Features shape: {X.shape}")
        print(f"   • Target shape: {y.shape}")
        print(f"   • Data source: {data_source}")
        print(f"   • Missing values in features: {X.isnull().sum().sum()}")
        print(f"   • Target distribution:")
        target_dist = y.value_counts().to_dict()
        for class_name, count in target_dist.items():
            print(f"     - Class {class_name}: {count} ({count/len(y)*100:.1f}%)")
        
        results = {
            'status': 'success',
            'data_source': data_source,
            'shape': X.shape,
            'target_distribution': target_dist,
            'missing_values': X.isnull().sum().to_dict(),
            'execution_time': step_time
        }
        
        print(f"\n⏱️  Step completed in {step_time:.2f} seconds")
        pipeline.results['data_loading'] = results
        
        return X, y, results
        
    except Exception as e:
        error_msg = f"Data loading failed: {e}"
        print(f"❌ {error_msg}")
        pipeline.results['data_loading'] = {
            'status': 'failed',
            'error': str(e),
            'execution_time': time.time() - step_start
        }
        raise RuntimeError(error_msg) from e

# Execute data loading
X, y, data_results = run_data_loading(pipeline)


In [ ]:
def run_preprocessing(pipeline, X: pd.DataFrame, y: pd.Series, skip_preprocessing: bool = False):
    """Preprocess features and split data."""
    print("=" * 50)
    print("STEP 3: DATA PREPROCESSING")
    print("=" * 50)
    
    try:
        step_start = time.time()
        
        if skip_preprocessing:
            print("⏭️  Skipping preprocessing (using data as-is)")
            X_processed = X
            preprocessing_steps = ["skipped"]
        else:
            print("🔄 Applying preprocessing pipeline...")
            X_processed, preprocessing_steps = preprocess_features(X, pipeline.config)
            print(f"✅ Preprocessing steps applied: {preprocessing_steps}")
        
        # Split data
        print("✂️  Splitting data into train/validation/test sets...")
        data_splits = split_data(X_processed, y, pipeline.config)
        
        step_time = time.time() - step_start
        
        # Summary of preprocessing results
        print(f"\n📈 Preprocessing Summary:")
        print(f"   • Original shape: {X.shape}")
        print(f"   • Processed shape: {X_processed.shape}")
        print(f"   • Features added/removed: {X_processed.shape[1] - X.shape[1]}")
        print(f"   • Training set: {data_splits['X_train'].shape[0]} samples")
        print(f"   • Validation set: {data_splits.get('X_val', pd.DataFrame()).shape[0]} samples")
        print(f"   • Test set: {data_splits['X_test'].shape[0]} samples")
        
        # Display feature names if they've changed significantly
        if X_processed.shape[1] != X.shape[1]:
            print(f"   • New features: {list(X_processed.columns)[:10]}..." if len(X_processed.columns) > 10 else f"   • All features: {list(X_processed.columns)}")
        
        results = {
            'status': 'success',
            'preprocessing_steps': preprocessing_steps,
            'original_shape': X.shape,
            'processed_shape': X_processed.shape,
            'train_size': data_splits['X_train'].shape[0],
            'val_size': data_splits.get('X_val', pd.DataFrame()).shape[0],
            'test_size': data_splits['X_test'].shape[0],
            'execution_time': step_time
        }
        
        print(f"\n⏱️  Step completed in {step_time:.2f} seconds")
        pipeline.results['preprocessing'] = results
        
        return data_splits, results
        
    except Exception as e:
        error_msg = f"Preprocessing failed: {e}"
        print(f"❌ {error_msg}")
        pipeline.results['preprocessing'] = {
            'status': 'failed',
            'error': str(e),
            'execution_time': time.time() - step_start
        }
        raise RuntimeError(error_msg) from e

# Execute preprocessing
data_splits, preprocessing_results = run_preprocessing(pipeline, X, y)
